In [ ]:
!pip install numpy torch pandas torchvision

In [ ]:
import numpy as np
import torch.nn as nn
import torch
import time
import pandas as pd
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

: 

In [ ]:
train_transforms = transforms.Compose([
    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2023, 0.1994, 0.2010)
    )
])

train_dataset = datasets.CIFAR10(root='./data', train=True, download=False, transform=train_transforms)
train_loader = DataLoader(
    train_dataset,
    batch_size=512,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

test_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2023, 0.1994, 0.2010)
    )
])

test_loader = DataLoader(
    datasets.CIFAR10(root='./data', train=False, download=False, transform=test_transforms),
    batch_size=512,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

print("Train samples: ", len(train_dataset))
print("Test samples: ", len(test_loader.dataset))

In [ ]:
class MbNetBlock(nn.Module):
  def __init__(self, in_ch, out_ch, stride):
    super(MbNetBlock, self).__init__()
    self.deepwise = nn.Conv2d(in_ch, in_ch, 3, stride, 1, groups=in_ch, bias=False)
    self.bn1 = nn.BatchNorm2d(in_ch)
    self.relu1 = nn.ReLU()
    self.pointwise = nn.Conv2d(in_ch, out_ch, 1, 1, 0, bias=False)
    self.bn2 = nn.BatchNorm2d(out_ch)
    self.relu2 = nn.ReLU()

  def forward(self, x):
    x = self.deepwise(x)
    x = self.bn1(x)
    x = self.relu1(x)
    x = self.pointwise(x)
    x = self.bn2(x)
    x = self.relu2(x)
    return x

class MobileNet(nn.Module):
  def __init__(self, num_classes=10, p=0.3):
    super(MobileNet, self).__init__()
    self.conv = nn.Sequential(
        nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1, bias=False),
        nn.BatchNorm2d(16),
        nn.ReLU()
    )
    self.features = nn.Sequential(
        MbNetBlock(16, 32, 2),
        MbNetBlock(32, 64, 2),
        MbNetBlock(64, 128, 2),
        MbNetBlock(128, 256, 2)
        )
    self.gap = nn.AdaptiveAvgPool2d((1,1))
    self.flatten = nn.Flatten()
    self.dropout = nn.Dropout(p)
    self.fc = nn.Linear(256, num_classes)

  def forward(self, x):
    x = self.conv(x)     
    x = self.features(x)
    x = self.gap(x)
    x = self.flatten(x)
    x = self.dropout(x)   
    x = self.fc(x)
    return x

In [ ]:
def evaluate(model, loader, criterion, device):
    model.eval()

    loss_sum = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            loss_sum += loss.item()

            _, pred = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (pred == labels).sum().item()

    return (loss_sum / len(loader), 100 * correct / total)

def train_loop(model, train_loader, val_loader, criterion, optimizer, device, epochs, save_path="best_model.pth", scheduler=None):
  model.to(device)

  history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
        "lr": []
  }

  best_val_acc = 0.0
  for ep in range (epochs):

    start_time = time.time()

    model.train()
    train_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
      images = images.to(device)
      labels = labels.to(device)

      optimizer.zero_grad()
      outputs = model(images)

      loss = criterion(outputs, labels)
      loss.backward()
      optimizer.step()
      train_loss += loss.item()

      _, predicted = torch.max(outputs, 1)
      total += labels.size(0)
      correct += (predicted == labels).sum().item()

    train_loss = train_loss / len(train_loader)
    train_acc = 100 * correct / total
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    current_lr = optimizer.param_groups[0]['lr']

    if scheduler is not None:
        scheduler.step()

    epoch_time = time.time() - start_time

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["lr"].append(current_lr)

    if val_acc > best_val_acc:
        best_val_acc = val_acc

        torch.save({
          "epoch": ep + 1,
          "model_state_dict": model.state_dict(),
          "optimizer_state_dict": optimizer.state_dict(),
          "best_val_acc": best_val_acc
        }, save_path)

        flag = " <-- BEST SAVED"

    else:
        flag = ""

    print(f"Epoch [{ep+1:03d}/{epochs}] | "
          f"Train Loss: {train_loss:.4f} | "
          f"Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} | "
          f"Val Acc: {val_acc:.4f} | "
          f"LR: {current_lr:.6f} | "
          f"Time: {epoch_time:.2f}s"
          f"{flag}"
    )
  print(f"\nBest Validation Accuracy = {best_val_acc:.4f}")
  return history


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device: ", device.type)

In [ ]:
model = MobileNet()
criterion = torch.nn.CrossEntropyLoss()
base_lr = 0.001 * 4 
optimizer = torch.optim.Adam(model.parameters(), lr=base_lr)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100, eta_min=base_lr * 0.01)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Base LR: {base_lr}")


In [ ]:
history = train_loop(
    model=model,
    train_loader=train_loader,
    val_loader=test_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=100,
    save_path="best_model.pth",
    scheduler=scheduler
)


In [ ]:
class STEQuantize(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input, scale, zero_point, quant_min=-128, quant_max=127):
        transformed = torch.round(input / scale) + zero_point
        clamped = torch.clamp(transformed, quant_min, quant_max)
        output = (clamped - zero_point) * scale
        return output

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output, None, None, None, None

class FakeQuant(nn.Module):
    def __init__(self, num_bits=8, is_activation=False, momentum=0.1):
        super().__init__()
        self.num_bits = num_bits
        self.qmin = -(2 ** (num_bits - 1))
        self.qmax = (2 ** (num_bits - 1)) - 1
        self.is_activation = is_activation
        self.momentum = momentum
        self.register_buffer('running_max', torch.tensor(0.0))

    def forward(self, x):
        if self.training:
            current_max = x.detach().abs().max()
            if self.is_activation:
                if self.running_max == 0.0:
                    self.running_max.copy_(current_max)
                else:
                    self.running_max.copy_((1 - self.momentum) * self.running_max + self.momentum * current_max)
                xmax = self.running_max
            else:
                xmax = current_max
        else:
            if self.is_activation:
                xmax = self.running_max
            else:
                xmax = x.detach().abs().max()

        scale = xmax / self.qmax
        scale = torch.clamp(scale, min=1e-8)
        zero_point = torch.zeros(1, device=x.device)
        
        return STEQuantize.apply(x, scale, zero_point, self.qmin, self.qmax)

In [ ]:
class QATConv2d(nn.Module):
    def __init__(self, in_ch, out_ch, k_size, stride=1, padding=0, groups=1, bias=False, w_bits=8, a_bits=8):
        super(QATConv2d, self).__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, k_size, stride=stride, padding=padding, groups=groups, bias=bias)
        self.w_quant = FakeQuant(num_bits=w_bits, is_activation=False)
        self.a_quant = FakeQuant(num_bits=a_bits, is_activation=True)

    def forward(self, x):
        x = self.a_quant(x)
        w = self.w_quant(self.conv.weight)
        return nn.functional.conv2d(x, w, self.conv.bias, stride=self.conv.stride, padding=self.conv.padding, groups=self.conv.groups)

class QATLinear(nn.Module):
    def __init__(self, in_ch, out_ch, w_bits=8, a_bits=8):
        super().__init__()
        self.fc = nn.Linear(in_ch, out_ch)
        self.w_quant = FakeQuant(num_bits=w_bits, is_activation=False)
        self.a_quant = FakeQuant(num_bits=a_bits, is_activation=True)

    def forward(self, x):
        x = self.a_quant(x)
        w = self.w_quant(self.fc.weight)
        return nn.functional.linear(x, w, self.fc.bias)

In [ ]:
class QATMbNetBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride):
        super(QATMbNetBlock, self).__init__()
        self.deepwise = QATConv2d(in_ch, in_ch, 3, stride, 1, groups=in_ch, bias=False)
        self.bn1 = nn.BatchNorm2d(in_ch)
        self.relu1 = nn.ReLU()
        self.pointwise = QATConv2d(in_ch, out_ch, 1, 1, 0, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.relu2 = nn.ReLU()

    def forward(self, x):
        x = self.deepwise(x)
        x = self.bn1(x)
        x = self.relu1(x)
        x = self.pointwise(x)
        x = self.bn2(x)
        x = self.relu2(x)
        return x

class QATMobileNet(nn.Module):
    def __init__(self, num_classes=10, p=0.3):
        super(QATMobileNet, self).__init__()
        self.conv = nn.Sequential(
            QATConv2d(3, 16, k_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU()
        )
        self.features = nn.Sequential(
            QATMbNetBlock(16, 32, 2),
            QATMbNetBlock(32, 64, 2),
            QATMbNetBlock(64, 128, 2),
            QATMbNetBlock(128, 256, 2)
        )
        self.gap = nn.AdaptiveAvgPool2d((1,1))
        self.flatten = nn.Flatten()
        self.dropout = nn.Dropout(p)
        self.fc = QATLinear(256, num_classes)
        
    def forward(self, x):
        x = self.conv(x)       
        x = self.features(x)
        x = self.gap(x)
        x = self.flatten(x)
        x = self.dropout(x)   
        x = self.fc(x)
        return x

In [ ]:
qat_model = QATMobileNet().to(device)

fp32_ckpt = torch.load("best_model.pth", map_location=device)
old_state_dict = fp32_ckpt["model_state_dict"]

new_state_dict = {}

for k, v in old_state_dict.items():
    new_k = k
    
    if k.startswith('fc.'):
        new_k = k.replace('fc.', 'fc.fc.', 1)
        
    elif k.endswith('.weight') or k.endswith('.bias'):
        if 'bn' not in k and 'conv.1' not in k:
            if k.endswith('.weight'):
                new_k = k.replace('.weight', '.conv.weight')
            elif k.endswith('.bias'):
                new_k = k.replace('.bias', '.conv.bias')
                
    new_state_dict[new_k] = v

qat_model.load_state_dict(new_state_dict, strict=False)

print("Load complete!")

In [ ]:
qat_base_lr = 0.0001 * 4 
optimizer_qat = torch.optim.Adam(qat_model.parameters(), lr=qat_base_lr)
scheduler_qat = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_qat, T_max=100, eta_min=qat_base_lr * 0.01)

history_qat = train_loop(
    model=qat_model,
    train_loader=train_loader,
    val_loader=test_loader,
    criterion=criterion,
    optimizer=optimizer_qat,
    device=device,
    epochs=100,
    save_path="best_qat_model.pth",
    scheduler=scheduler_qat
)
